# Faruq-v3 WAV_L1 paired multiseed confirmation — Kaggle

Attach only the private `faruq-v3-experiment-core-v1` dataset. This notebook reuses frozen WAV_L1 seed 42 repository evidence, trains only seeds 123 and 2026 from seed-matched D0 checkpoints, computes the pre-frozen three-seed decision, and keeps the Faruq locked test closed.

If a previous incomplete Kaggle Saved Version of this same notebook is attached as input, compatible run directories are restored automatically from their frozen run contracts.

In [ ]:
from pathlib import Path
INPUT=Path('/kaggle/input'); WORK=Path('/kaggle/working')
if not INPUT.is_dir() or not WORK.is_dir(): raise RuntimeError('Notebook ini Kaggle-only.')
manifests=sorted(p for p in INPUT.rglob('af2_spectral_kaggle_manifest.json') if p.is_file())
if len(manifests)!=1: raise FileNotFoundError(f'STOP CEPAT: core manifest={manifests}')
for name in ('D0_seed123_best.pt','D0_seed2026_best.pt','af2_igem_paired_confirmation.json'):
    hits=sorted(p for p in INPUT.rglob(name) if p.is_file())
    if len(hits)!=1: raise FileNotFoundError(f'STOP CEPAT: {name}={hits}')
print('FAST INPUT PREFLIGHT PASS')
print('CORE:',manifests[0])
print('Seed42 WAV_L1 akan dibaca dari frozen repository evidence; locked test tidak digunakan.')


In [ ]:
import importlib,json,os,shutil,subprocess,sys,time,hashlib
from pathlib import Path
WORK=Path('/kaggle/working'); INPUT=Path('/kaggle/input')
REPO=WORK/'coffee-bean-detection'
OUT=WORK/'wav-l1-paired-confirmation-v1'
BRANCH='agent/wav1-mechanism-factorization'
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    r=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if r.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt==2: raise RuntimeError('git clone gagal tiga kali')
    time.sleep(2)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
for m in list(sys.modules):
    if m=='coffee_detector' or m.startswith('coffee_detector.'): sys.modules.pop(m,None)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
print('COMMIT:',COMMIT)
from coffee_detector.experiments.prepare_af2_spectral_kaggle import prepare_af2_spectral_kaggle_input
from coffee_detector.experiments.run_faruq_v3_wav_l1_confirmation_seed import CONFIG, PROTOCOL, SEED42_EVIDENCE
DATA,A,CONTRACT=prepare_af2_spectral_kaggle_input(INPUT,WORK)
assert CONTRACT['decision']=='PASS' and CONTRACT['test_images_accessed'] is False
D123=A['D0_seed123_best.pt']; D2026=A['D0_seed2026_best.pt']; D0REF=A['af2_igem_paired_confirmation.json']
if not SEED42_EVIDENCE.is_file(): raise FileNotFoundError(SEED42_EVIDENCE)
if not PROTOCOL.is_file(): raise FileNotFoundError(PROTOCOL)
seed42=json.loads(SEED42_EVIDENCE.read_text(encoding='utf-8'))
assert seed42['arm']=='WAV_L1' and seed42['seed']==42 and seed42['evaluation_split']=='val' and seed42['test_images_accessed'] is False
OUT.mkdir(exist_ok=True); (OUT/'val_reports').mkdir(parents=True,exist_ok=True)
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
print('DATA:',DATA)
print('D0 seed123:',D123)
print('D0 seed2026:',D2026)
print('FROZEN SEED42:',SEED42_EVIDENCE)
print('FROZEN PROTOCOL:',PROTOCOL)


In [ ]:
# Optional resume from an attached previous Kaggle Saved Version.
def restore(seed,d0):
    expected={
      'format':'coffee_detector.wav_l1_confirmation.run_contract.v1',
      'arm':'WAV_L1',
      'seed':seed,
      'config_sha256':sha(CONFIG),
      'protocol_sha256':sha(PROTOCOL),
      'seed42_evidence_sha256':sha(SEED42_EVIDENCE),
      'd0_checkpoint_sha256':sha(d0),
      'epochs':50,
      'evaluation_split':'val',
      'test_images_accessed':False,
    }
    matches=[]
    for cp in INPUT.rglob('run_contract.json'):
        try: payload=json.loads(cp.read_text(encoding='utf-8'))
        except Exception: continue
        if payload==expected: matches.append(cp.parent)
    if not matches: return None
    if len(matches)!=1: raise RuntimeError(f'Resume WAV_L1 seed {seed} ambigu: {matches}')
    dst=OUT/'WAV_L1'/f'WAV_L1_seed{seed}'
    if not dst.exists(): dst.parent.mkdir(parents=True,exist_ok=True); shutil.copytree(matches[0],dst)
    return dst
print('RESTORE 123:',restore(123,D123))
print('RESTORE 2026:',restore(2026,D2026))


In [ ]:
# Train/evaluate only the two authorized confirmation seeds.
LOG=OUT/'wav_l1_paired_confirmation.log'
def run_seed(seed,d0):
    result_path=OUT/'val_reports'/f'WAV_L1_seed{seed}_result.json'
    cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_wav_l1_confirmation_seed',
      '--data-root',str(DATA),'--grouped-summary',str(DATA/'faruq_grouped_summary.json'),
      '--d0-checkpoint',str(d0),'--output-root',str(OUT),'--seed',str(seed),'--device','0','--authorize-training']
    if not result_path.is_file():
        with LOG.open('a',encoding='utf-8',buffering=1) as stream:
            process=subprocess.Popen(cmd,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
            seen=-1
            while process.poll() is None:
                csv=OUT/'WAV_L1'/f'WAV_L1_seed{seed}'/'results.csv'
                n=max(0,len(csv.read_text(errors='replace').splitlines())-1) if csv.is_file() else 0
                if n!=seen: print(f'WAV_L1 seed {seed}: {n}/50 epoch | log={LOG}',flush=True); seen=n
                time.sleep(120)
            code=process.wait()
        if code:
            tail='\n'.join(LOG.read_text(errors='replace').splitlines()[-220:]) if LOG.is_file() else '<log tidak ditemukan>'
            raise RuntimeError(f'WAV_L1 seed {seed} gagal: returncode={code}\n--- LOG TAIL ---\n{tail}')
    if not result_path.is_file(): raise FileNotFoundError(result_path)
    payload=json.loads(result_path.read_text(encoding='utf-8'))
    assert payload['seed']==seed and payload['arm']=='WAV_L1' and payload['evaluation_split']=='val' and payload['test_images_accessed'] is False
    print(f'=== SEED {seed} COMPLETE ===')
    print(json.dumps(payload['metrics'],indent=2))
    return result_path
R123=run_seed(123,D123)
R2026=run_seed(2026,D2026)


In [ ]:
# Apply the pre-frozen paired three-seed decision gate.
DECISION=OUT/'val_reports'/'wav_l1_paired_confirmation.json'
cmd=[sys.executable,'-u','-m','coffee_detector.experiments.run_faruq_v3_wav_l1_paired_decision',
 '--seed42-evidence',str(SEED42_EVIDENCE),'--seed123-result',str(R123),'--seed2026-result',str(R2026),
 '--af2-igem-reference',str(D0REF),'--output',str(DECISION)]
subprocess.run(cmd,cwd=REPO,check=True)
result=json.loads(DECISION.read_text(encoding='utf-8'))
assert result['test_opened'] is False and result['test_images_accessed'] is False
print('=== WAV_L1 THREE-SEED DECISION ===')
for seed,row in result['per_seed'].items():
    print('seed',seed,'D0FT=',row['D0FT'],'WAV_L1=',row['WAV_L1'])
print('\nAGGREGATE:')
for metric,row in result['aggregate'].items(): print(metric,row)
print('\nCRITERIA:',result['criteria'])
print('DECISION:',result['decision'])
print('NEXT:',result['next_action'])


In [ ]:
# Build a compact Saved-Version handoff. No locked-test material is included.
HANDOFF=WORK/'wav-l1-paired-confirmation-handoff'
shutil.rmtree(HANDOFF,ignore_errors=True); (HANDOFF/'val_reports').mkdir(parents=True)
for p in (OUT/'val_reports').glob('*.json'): shutil.copy2(p,HANDOFF/'val_reports'/p.name)
shutil.copy2(SEED42_EVIDENCE,HANDOFF/SEED42_EVIDENCE.name)
for seed in (123,2026):
    best=OUT/'WAV_L1'/f'WAV_L1_seed{seed}'/'weights/best.pt'
    contract=OUT/'WAV_L1'/f'WAV_L1_seed{seed}'/'run_contract.json'
    if contract.is_file():
        dst=HANDOFF/'resume'/f'WAV_L1_seed{seed}'
        dst.parent.mkdir(parents=True,exist_ok=True)
        shutil.copytree(contract.parent,dst,dirs_exist_ok=True)
    if result['decision']=='PASS' and best.is_file(): shutil.copy2(best,HANDOFF/f'WAV_L1_seed{seed}_best.pt')
manifest={
 'format':'coffee_detector.wav_l1_confirmation.kaggle_handoff.v1',
 'commit':COMMIT,'seeds':[42,123,2026],'decision':result['decision'],
 'seed42_source':'repository_frozen_evidence','test_images_accessed':False,
 'note':'Save Version. Reattach this output only if resume/review is needed.'}
(HANDOFF/'handoff_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n',encoding='utf-8')
if DATA.exists(): shutil.rmtree(DATA)
if REPO.exists(): shutil.rmtree(REPO)
print('HANDOFF READY:',HANDOFF)
print('SAVE VERSION. Kirim DECISION + AGGREGATE ke chat. Locked test tetap tertutup.')
